# Batch Model Monitoring Demo - Setup Model Monitor

Creates a Model Monitor for the CHURN_PREDICTOR model that:
- Watches the SCORING_DATA table for new predictions
- Compares feature distributions against BASELINE_DATA to detect drift
- Segments results by PLAN_TYPE and CONTRACT_TYPE
- Refreshes every hour with 1-day aggregation windows

After this notebook runs, the monitor appears in Snowsight under:
**AI & ML → Models → CHURN_PREDICTOR → Monitors**

In [ ]:
%%sql -r df_ctx
USE DATABASE ML_DEMOS;
USE SCHEMA BATCH_MONITORING;
USE WAREHOUSE ML_DEMO_WH;

In [ ]:
%%sql -r df_monitor
-- Create the model monitor
CREATE OR REPLACE MODEL MONITOR CHURN_MONITOR WITH
    MODEL = CHURN_PREDICTOR
    VERSION = 'V1'
    FUNCTION = 'PREDICT_PROBA'
    SOURCE = ML_DEMOS.BATCH_MONITORING.SCORING_DATA
    WAREHOUSE = ML_DEMO_WH
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = PREDICTION_TS
    BASELINE = ML_DEMOS.BATCH_MONITORING.BASELINE_DATA
    ID_COLUMNS = ('ID')
    PREDICTION_SCORE_COLUMNS = ('PREDICTION_SCORE')
    SEGMENT_COLUMNS = ('PLAN_TYPE', 'CONTRACT_TYPE');

In [ ]:
%%sql -r df_describe
-- Inspect monitor configuration
DESCRIBE MODEL MONITOR CHURN_MONITOR;

In [ ]:
%%sql -r df_show
-- List all monitors in this schema
SHOW MODEL MONITORS IN SCHEMA ML_DEMOS.BATCH_MONITORING;

## What happens next

The monitor is now active and will:
1. **Refresh** every hour — query SCORING_DATA for new rows since last refresh
2. **Aggregate** into 1-day windows — compute distribution statistics per day
3. **Compare** against BASELINE_DATA — calculate drift metrics (PSI, Jensen-Shannon)
4. **Segment** by PLAN_TYPE and CONTRACT_TYPE — separate metrics per segment

### View in Snowsight
Navigate to: **AI & ML → Models → CHURN_PREDICTOR → Monitors → CHURN_MONITOR**

The dashboard shows:
- Prediction score distribution over time
- Feature drift (PSI) per feature per day
- Row volume per aggregation window
- Segment-level breakdowns

### Next steps
Run `05_simulate_drift.ipynb` to insert progressively drifted data, then
`06_observe_and_query.ipynb` to query the resulting metrics programmatically.